## ✅ STEP 1 — Import Libraries

In [23]:
import pandas as pd
import glob
import os

## ✅ STEP 2 — Define Data Path

In [24]:
DATA_PATH = "Telangana_datasets/"

## ✅ STEP 3 — Load & Combine Card Status Files

In [25]:
card_files = glob.glob(DATA_PATH + "fpshop-card-status*.csv")

card_df = pd.concat(
    [pd.read_csv(f) for f in card_files],
    ignore_index=True
)

print("Card Status Shape:", card_df.shape)

Card Status Shape: (517516, 28)


In [26]:
card_df.columns

Index(['distCode', 'distName', 'officeCode', 'officeName', 'shopNo', 'month',
       'year', 'rcNfsaAay', 'unitsNfsaAay', 'rcNfsaPhh', 'unitsNfsaPhh',
       'totalRcNfsa', 'totalUnitsNfsa', 'rcStateAay', 'unitsStateAay',
       'rcStatePhh', 'unitsStatePhh', 'rcStateAap', 'unitsStateAap',
       'totalRcState', 'totalUnitsState', 'totalRcs', 'totalUnits',
       'cardTypeId', 'units', 'gasCylinders', 'mroAsoApprDate',
       'cardPoolType'],
      dtype='object')

## ✅ STEP 4 — Load & Combine Transaction Files

In [27]:
trans_files = glob.glob(DATA_PATH + "shop-wise-trans-details*.csv")

trans_df = pd.concat(
    [pd.read_csv(f) for f in trans_files],
    ignore_index=True
)

print("Transactions Shape:", trans_df.shape)

Transactions Shape: (499648, 19)


In [28]:
trans_df.columns

Index(['distCode', 'distName', 'officeCode', 'officeName', 'shopNo', 'month',
       'year', 'noOfRcs', 'noOfTrans', 'riceAfsc', 'riceFsc', 'riceAap',
       'wheat', 'sugar', 'rgdal', 'kerosene', 'totalAmount', 'salt',
       'otherShopTransCnt'],
      dtype='object')

## ✅ STEP 5 — Load & Combine Shop Status Files (Geo Info)

In [29]:
shop_files = glob.glob(DATA_PATH + "shop-status-details*.csv")

shop_df = pd.concat(
    [pd.read_csv(f) for f in shop_files],
    ignore_index=True
)

print("Shop Status Shape:", shop_df.shape)

Shop Status Shape: (17434, 11)


In [30]:
shop_df.columns

Index(['distCode', 'distName', 'officeCode', 'officeName', 'shopNo', 'address',
       'longitude', 'latitude', 'fpsStatus', 'fpsType', 'dateTime'],
      dtype='object')

## ✅ STEP 6 — Clean Before Merge
Remove duplicates:

In [31]:
card_df.drop_duplicates(inplace=True)
trans_df.drop_duplicates(inplace=True)
shop_df.drop_duplicates(inplace=True)
print("Card Status Shape:", card_df.shape)
print("Transactions Shape:", trans_df.shape)
print("Shop Status Shape:", shop_df.shape)

Card Status Shape: (517516, 28)
Transactions Shape: (499648, 19)
Shop Status Shape: (17434, 11)


In [32]:
join_cols_time = ["distCode", "shopNo", "month", "year"]
join_cols_static = ["distCode", "shopNo"]

for col in join_cols_time:
    card_df[col] = card_df[col].astype(str)
    trans_df[col] = trans_df[col].astype(str)

for col in join_cols_static:
    shop_df[col] = shop_df[col].astype(str)

## ✅ STEP 7 — First Merge (Card + Transactions)

In [33]:
merged_time = pd.merge(
    card_df,
    trans_df,
    on=join_cols_time,
    how="inner"
)

print("After Time Merge:", merged_time.shape)

After Time Merge: (499494, 43)


## ✅ STEP 8 — Second Merge (Add Shop Metadata)

In [34]:
master_df = pd.merge(
    merged_time,
    shop_df,
    on=join_cols_static,
    how="left"
)

print("Final Master Dataset Shape:", master_df.shape)

Final Master Dataset Shape: (499494, 52)


## ✅ STEP 9 — Validation Checks (Important)

In [35]:
# Check duplicates:

print("Duplicate Rows:", master_df.duplicated().sum())

Duplicate Rows: 0


In [36]:
# Check null summary:

print(master_df.isnull().sum().sort_values(ascending=False).head(15))


mroAsoApprDate    499494
cardTypeId        499494
cardPoolType      499494
latitude            4794
longitude           4794
address             4614
fpsType             4266
fpsStatus           4266
officeName          4266
officeCode          4266
distName            4266
dateTime            4266
totalUnitsNfsa         0
rcStateAay             0
riceFsc                0
dtype: int64


In [37]:
# Check sample:

master_df.head()

,distCode,distName_x,officeCode_x,officeName_x,shopNo,month,year,rcNfsaAay,unitsNfsaAay,rcNfsaPhh,...,otherShopTransCnt,distName,officeCode,officeName,address,longitude,latitude,fpsStatus,fpsType,dateTime
0,532,Adilabad,532001,Talamadugu,1901001,7,2024,38,125,920,...,36,Adilabad,532001.0,Talamadugu,DR Depo TALAMADUGU Talamadugu,78.391212,19.641643,Active,Normal Shop,2025-07-03 14:43:54.124224+05:30
1,532,Adilabad,532001,Talamadugu,1901002,7,2024,20,70,258,...,28,Adilabad,532001.0,Talamadugu,Sunkari Ramesh Sunkidi,78.422012,19.669093,Active,Normal Shop,2025-07-03 14:43:54.124224+05:30
2,532,Adilabad,532001,Talamadugu,1901003,7,2024,17,60,158,...,22,Adilabad,532001.0,Talamadugu,Sakipelli Gajanan Umdam,78.427186,19.664645,Active,Normal Shop,2025-07-03 14:43:54.124224+05:30
3,532,Adilabad,532001,Talamadugu,1901004,7,2024,14,45,190,...,12,Adilabad,532001.0,Talamadugu,Rebbathi Rakesh Lingi,78.381016,19.696032,Active,Normal Shop,2025-07-03 14:43:54.124224+05:30
4,532,Adilabad,532001,Talamadugu,1901005,7,2024,155,520,530,...,30,Adilabad,532001.0,Talamadugu,T Manohar Kuchulapur,78.351118,19.697063,Active,Normal Shop,2025-07-03 14:43:54.124224+05:30


In [38]:
master_df.columns

Index(['distCode', 'distName_x', 'officeCode_x', 'officeName_x', 'shopNo',
       'month', 'year', 'rcNfsaAay', 'unitsNfsaAay', 'rcNfsaPhh',
       'unitsNfsaPhh', 'totalRcNfsa', 'totalUnitsNfsa', 'rcStateAay',
       'unitsStateAay', 'rcStatePhh', 'unitsStatePhh', 'rcStateAap',
       'unitsStateAap', 'totalRcState', 'totalUnitsState', 'totalRcs',
       'totalUnits', 'cardTypeId', 'units', 'gasCylinders', 'mroAsoApprDate',
       'cardPoolType', 'distName_y', 'officeCode_y', 'officeName_y', 'noOfRcs',
       'noOfTrans', 'riceAfsc', 'riceFsc', 'riceAap', 'wheat', 'sugar',
       'rgdal', 'kerosene', 'totalAmount', 'salt', 'otherShopTransCnt',
       'distName', 'officeCode', 'officeName', 'address', 'longitude',
       'latitude', 'fpsStatus', 'fpsType', 'dateTime'],
      dtype='object')

In [39]:
(master_df['distName_x'] == master_df['distName_y']).all()


np.True_

In [40]:
(master_df['distName_x'] == master_df['distName']).all()

np.False_

In [41]:
master_df[master_df['distName_x'] != master_df['distName']]

,distCode,distName_x,officeCode_x,officeName_x,shopNo,month,year,rcNfsaAay,unitsNfsaAay,rcNfsaPhh,...,otherShopTransCnt,distName,officeCode,officeName,address,longitude,latitude,fpsStatus,fpsType,dateTime
41,532,Adilabad,532004,Jainad,1904016,7,2024,50,156,517,...,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
42,532,Adilabad,532004,Jainad,1904017,7,2024,8,24,198,...,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
43,532,Adilabad,532004,Jainad,1904018,7,2024,14,42,278,...,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44,532,Adilabad,532004,Jainad,1904019,7,2024,55,160,519,...,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47,532,Adilabad,532004,Jainad,1904022,7,2024,7,28,257,...,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497737,939,Wanaparthy,939005,Gopalpeta,4205026,2,2023,15,36,112,...,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
497869,939,Wanaparthy,939011,Revally,4211010,2,2023,23,53,177,...,118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
497870,939,Wanaparthy,939011,Revally,4211011,2,2023,28,70,131,...,151,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
497871,939,Wanaparthy,939011,Revally,4211012,2,2023,19,45,142,...,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
master_df[
    master_df['distName_x'] != master_df['distName']
][['distName_x', 'distName']].drop_duplicates()

,distName_x,distName
41,Adilabad,NaN
7397,Jagityal,NaN
8598,Kamareddy,NaN
11118,Nagarkarnool,NaN
12076,Peddapalli,NaN
12525,Sangareddy,NaN
13403,Siddipet,NaN
15474,Wanaparthy,NaN
52138,Nizamabad,NaN
56513,Nalgonda,NaN


In [43]:
master_df.drop(
    columns=[
        'distName_y',
        'distName',
        'officeCode_x',
        'officeCode_y',
        'officeName_x',
        'officeName_y'
    ],
    inplace=True
)

master_df.rename(columns={'distName_x': 'distName'}, inplace=True)

## ✅ STEP 10 — Save Master Dataset

In [44]:
master_df.to_csv("master_pds_dataset.csv", index=False)
print("Master dataset saved successfully.")

Master dataset saved successfully.
